In [ ]:
import pandas as pd
import anndata2ri
import rpy2.rinterface_lib.callbacks
import logging
import sys
import os
pd.DataFrame.iteritems = pd.DataFrame.items

anndata2ri.activate()
%reload_ext rpy2.ipython

In [ ]:

homeDir = os.getenv("HOME")

sys.path.insert(1, homeDir+"/utils/")


from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
from spatialUtils import *
from _DEAplots import *
from _Aggregation import *
from _plotting import *
import rapids_singlecell as rsc


In [ ]:
import ipynbname
import nbconvert.exporters
from nbconvert.preprocessors import TagRemovePreprocessor
import os


try:
    nb_name = ipynbname.name()
except:
    nb_name = "".join(os.path.basename(globals()['__vsc_ipynb_file__']))

print(nb_name)

# Importing count matrices

In [ ]:
%%R -o Counts_Ciceri -o Counts_Verrillo
library(dplyr)
library(tibble)
library(AnnotationDbi)
library(org.Hs.eg.db)
library(dplyr)
library(tibble)
library(AnnotationDbi)
library(org.Hs.eg.db)


Counts_Ciceri <- read.delim(
  "/data/projects/spatialTX/data/BulkSignature_RawData/GSE196073_raw_counts_GRCh38.p13_NCBI.tsv",
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

rownames(Counts_Ciceri) <- trimws(rownames(Counts_Ciceri))

colnames(Counts_Ciceri) <- dplyr::recode(
  colnames(Counts_Ciceri),
  "GSM5859021" = "CN1_ESC",
  "GSM5859022" = "CN1_NPC",
  "GSM5859023" = "CN1_d25",
  "GSM5859024" = "CN1_d50",
  "GSM5859025" = "CN1_d75",
  "GSM5859026" = "CN1_d100",
  "GSM5859027" = "CN3_ESC",
  "GSM5859028" = "CN3_NPC",
  "GSM5859029" = "CN3_d25",
  "GSM5859030" = "CN3_d50",
  "GSM5859031" = "CN3_d75",
  "GSM5859032" = "CN3_d100",
  "GSM5859033" = "CN5_ESC",
  "GSM5859034" = "CN5_NPC",
  "GSM5859035" = "CN5_d25",
  "GSM5859036" = "CN5_d50",
  "GSM5859037" = "CN5_d75",
  "GSM5859038" = "CN5_d100"
)

gene_ids <- rownames(Counts_Ciceri)

map <- AnnotationDbi::select(
  org.Hs.eg.db,
  keys = gene_ids,
  keytype = "ENTREZID",
  columns = "SYMBOL"
)

map <- map[!is.na(map$SYMBOL) & nzchar(map$SYMBOL), , drop = FALSE]

# keep first SYMBOL per ENTREZID
map1 <- map[!duplicated(map$ENTREZID), , drop = FALSE]
id2sym <- setNames(map1$SYMBOL, map1$ENTREZID)

sym <- unname(id2sym[gene_ids])
keep <- !is.na(sym)

Counts_Ciceri <- Counts_Ciceri[keep, , drop = FALSE]
sym <- sym[keep]

Counts_Ciceri <- Counts_Ciceri |>
  as.data.frame() |>
  tibble::rownames_to_column("ENTREZID")

Counts_Ciceri$SYMBOL <- sym

Counts_Ciceri <- Counts_Ciceri |>
  dplyr::select(-ENTREZID) |>
  dplyr::group_by(SYMBOL) |>
  dplyr::summarise(dplyr::across(everything(), ~ sum(.x, na.rm = TRUE)), .groups = "drop") |>
  as.data.frame()

rownames(Counts_Ciceri) <- Counts_Ciceri$SYMBOL
Counts_Ciceri$SYMBOL <- NULL

head(Counts_Ciceri)




Counts_Verrillo <- read.delim(
  "/data/projects/spatialTX/data/BulkSignature_RawData/rsem.merged.gene_counts.tsv",
  sep = "\t",
  row.names = 1,
  check.names = FALSE
)

rownames(Counts_Verrillo) <- trimws(rownames(Counts_Verrillo))

# strip ENSG version if present
ensg0 <- sub("\\..*$", "", rownames(Counts_Verrillo))

# map ENSG -> SYMBOL
map <- AnnotationDbi::select(
  org.Hs.eg.db,
  keys = unique(ensg0),
  keytype = "ENSEMBL",
  columns = "SYMBOL"
)

map <- map[!is.na(map$SYMBOL) & nzchar(map$SYMBOL), , drop = FALSE]

# keep first symbol per ENSG
map1 <- map[!duplicated(map$ENSEMBL), , drop = FALSE]
ensg2sym <- setNames(map1$SYMBOL, map1$ENSEMBL)

sym <- unname(ensg2sym[ensg0])
keep <- !is.na(sym)

Counts_Verrillo <- Counts_Verrillo[keep, , drop = FALSE]
sym <- sym[keep]

# sum many ENSGs mapping to the same symbol
Counts_Verrillo <- Counts_Verrillo |>
  as.data.frame() |>
  tibble::rownames_to_column("ENSEMBL")

Counts_Verrillo$SYMBOL <- sym

Counts_Verrillo <- Counts_Verrillo |>
  dplyr::select(-ENSEMBL) |>
  dplyr::group_by(SYMBOL) |>
  dplyr::summarise(
    dplyr::across(everything(), ~ sum(as.numeric(.x), na.rm = TRUE)),
    .groups = "drop"
  ) |>
  as.data.frame()

rownames(Counts_Verrillo) <- Counts_Verrillo$SYMBOL
Counts_Verrillo$SYMBOL <- NULL



In [ ]:
Counts_Ciceri

In [ ]:
Counts_Ciceri.columns = Counts_Ciceri.columns.str.replace(r"^GSM\d+_Sample_", "", regex=True)
Counts_Ciceri.columns = Counts_Ciceri.columns.str.replace(r"^GSM\d+_Sample_", "", regex=True)
Counts_Ciceri.columns = [col.replace("CN","REP") for col in Counts_Ciceri.columns]
Counts_Ciceri = Counts_Ciceri[[col for col in Counts_Ciceri.columns if  "_d75" in col ]]
Counts_Ciceri.columns = Counts_Ciceri.columns.str.replace(r"_d75", "_Neurons", regex=True)
# Counts_Ciceri.columns = Counts_Ciceri.columns.str.replace(r"_d50", "_IntermediateNeurons", regex=True)

# Counts_Ciceri.columns = Counts_Ciceri.columns.str.replace(r"_d100", "_LateNeurons", regex=True)
Counts_Ciceri.columns = [col.split("_")[1]+"_"+col.split("_")[0] for col in Counts_Ciceri.columns]
Counts_Ciceri.head()

In [ ]:
Counts_Verrillo

In [ ]:
Counts_Verrillo = Counts_Verrillo[~Counts_Verrillo.index.duplicated(keep="first")].copy()
Counts_Ciceri = Counts_Ciceri[~Counts_Ciceri.index.duplicated(keep="first")].copy()
CommonGenes = list(set(Counts_Ciceri.index.tolist()).intersection(Counts_Verrillo.index.tolist()))
Counts_Verrillo = Counts_Verrillo.loc[CommonGenes]
Counts_Ciceri = Counts_Ciceri.loc[CommonGenes]

Counts = pd.concat([Counts_Verrillo,Counts_Ciceri], axis = 1)
#Counts = Counts[[col for col in Counts.columns if "MB_" in col or "MELA_" in col or "NCRE_" in col]]



MD = pd.DataFrame(index=Counts.T.index)
MD["sample"] = MD.index.tolist()
MD["sample"]  = MD["sample"].map(str)
# MD["sample"]  = MD["sample"].map(str)   # forces python str (not numpy.str_)
MD["cluster"] = MD["sample"].str.split("_", expand=True)[0]
# MD.loc[MD["sample"].str.contains("MB_"),"cluster"] = "MB"
# MD.loc[MD["sample"].str.contains("MELA_"),"cluster"] = "MELA"
# MD.loc[MD["sample"].str.contains("NCRE_"),"cluster"] = "NCRE"
# MD.loc[(MD["sample"].str.contains("CN")) & (MD["sample"].str.contains("_d25")),"cluster"] = "EarlyCN"
# MD.loc[(MD["sample"].str.contains("CN")) & (MD["sample"].str.contains("_d")),"cluster"] = "CN"
# MD.loc[(MD["sample"].str.contains("CN")) & (MD["sample"].str.contains("_NPC")),"cluster"] = "NPC"

MD["cluster"]

MD["sample"] = MD["sample"].astype(str)

In [ ]:
MD

In [ ]:
tag = "_".join(MD.cluster.unique().tolist())

In [ ]:
f"./AggregatedCounts_{tag}.csv"

In [ ]:
Counts.to_csv(f"./AggregatedCounts_{tag}.csv")

In [ ]:
%%R -i MD -i Counts -o results  -o genes

library(edgeR)

# --- Align MD to Counts columns ---
stopifnot(nrow(MD) == ncol(Counts))
MD <- MD[match(colnames(Counts), MD$sample), , drop=FALSE]
stopifnot(all(MD$sample == colnames(Counts)))
rownames(MD) <- MD$sample

# --- Group factor ---
cluster <- factor(MD$cluster)
stopifnot(nlevels(cluster) > 1)

# --- DGEList ---
y <- DGEList(counts = Counts, samples = MD)
y$samples$cluster <- cluster


# --- Design (no intercept) ---
design <- model.matrix(~ 0 + cluster)
colnames(design) <- sub("^cluster", "", colnames(design))

# --- Filtering (pass design) ---
keep.genes <- filterByExpr(y, design = design)
y <- y[keep.genes, , keep.lib.sizes = FALSE]

# --- Normalization + fit ---
y <- calcNormFactors(y)
y <- estimateDisp(y, design, robust = TRUE)
fit <- glmQLFit(y, design, robust = TRUE)

genes <- rownames(y)  # <-- define genes AFTER filtering

# --- Contrasts: each cluster vs mean(other clusters) ---
ncls <- ncol(design)
contr <- matrix(-1/(ncls - 1), nrow=ncls, ncol=ncls,
                dimnames=list(colnames(design), colnames(design)))
diag(contr) <- 1

alpha <- 0.05
lfc   <- 0

results <- list()
calls   <- list()

for (nm in colnames(contr)) {
  message("Testing: ", nm)
  qlf <- glmQLFTest(fit, contrast = contr[, nm])

  tt <- topTags(qlf, n = Inf, sort.by = "logFC")$table
  tt$genes <- rownames(tt)              # keep genes as explicit column for Python
  results[[nm]] <- tt

  # calls: -1/0/+1 (down / ns / up) as a named vector
  call_vec <- ifelse(tt$FDR < alpha & abs(tt$logFC) >= lfc, sign(tt$logFC), 0)
  names(call_vec) <- rownames(tt)
  calls[[nm]] <- call_vec
}




In [ ]:
resultsDict = dict(zip([str(i) for i in list(results.names())], list(results.values())))
resultsDictExport = resultsDict.copy()

for k in list(resultsDictExport.keys()):
    df = resultsDictExport[k]
    df["celltype"] = k
    resultsDictExport[k] = df

combined = pd.concat(list(resultsDictExport.values()), ignore_index=True)
combined[combined["genes"] == "SOX10"]

In [ ]:
logFCmin = 0.1
FDRmax = 0.05

resultsDict = dict(zip([str(i) for i in list(results.names())], list(results.values())))
resultsDictExport = resultsDict.copy()

for k in list(resultsDictExport.keys()):
    df = resultsDictExport[k]
    df["celltype"] = k
    resultsDictExport[k] = df

combined = pd.concat(list(resultsDictExport.values()), ignore_index=True)

# count how many celltypes each gene is upregulated in
nUp = combined[
    (combined["logFC"] >= logFCmin) &
    (combined["FDR"] < FDRmax)
]["genes"].value_counts()

# assign counts
combined["N_celltypes_upregulated"] = combined["genes"].map(nUp).fillna(0).astype(int)

up = combined[
    (combined["logFC"] >= logFCmin) &
    (combined["FDR"] < FDRmax)
]

celltype_map = up.groupby("genes")["celltype"].apply(lambda x: ",".join(sorted(x)))

combined["Celltypes_upregulated"] = combined["genes"].map(celltype_map).fillna("NonDEG")


combined = combined[combined["FDR"] < FDRmax ].copy()
combined = combined[combined["logFC"] >= logFCmin ].copy()


saveDir = f"{nb_name}_signatures_{tag}"
os.makedirs(saveDir, exist_ok=True)

for celltype in combined["celltype"].unique():
    localDegs = combined[combined["celltype"] == celltype].copy()  
    localDegs.to_excel(os.path.join(saveDir, f"Signature_{celltype}_minlogFC{logFCmin}_maxFDR{FDRmax}.xlsx"), index=False)

In [ ]:
combined